# AI-Driven Employee Attrition Prediction System

###### ------> The model predicts the probability of attrition for each employee. Based on these probabilities, the top five high-risk employees were identified to support proactive HR intervention.

###### ------> I used Random Forest Model which gives accuracy around 87-88% (To be precise = 88.09 %)
###### ------> Although advanced models like XGBoost can slightly improve accuracy, the focus of this project is on interpretability and actionable HR insights rather than just maximizing accuracy.

In [4]:
import pandas as pd

In [5]:
df = pd.read_csv("HR-Employee-Attrition.csv")

In [8]:
df

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1465,36,No,Travel_Frequently,884,Research & Development,23,2,Medical,1,2061,...,3,80,1,17,3,3,5,2,0,3
1466,39,No,Travel_Rarely,613,Research & Development,6,1,Medical,1,2062,...,1,80,1,9,5,3,7,7,1,7
1467,27,No,Travel_Rarely,155,Research & Development,4,3,Life Sciences,1,2064,...,2,80,1,6,0,3,6,2,0,3
1468,49,No,Travel_Frequently,1023,Sales,2,3,Medical,1,2065,...,4,80,0,17,3,2,9,6,0,8


# -----------------------------------------------------------------------------------------
# 1 -- BASIC DATA CHECK

##### -- Number of employees (rows)
##### -- Number of columns

In [12]:
df.shape

(1470, 35)

## --------------------------------------------------------------------------------------
##### -- Column names
##### -- Data types
##### -- Missing values

In [15]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 35 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   Age                       1470 non-null   int64 
 1   Attrition                 1470 non-null   object
 2   BusinessTravel            1470 non-null   object
 3   DailyRate                 1470 non-null   int64 
 4   Department                1470 non-null   object
 5   DistanceFromHome          1470 non-null   int64 
 6   Education                 1470 non-null   int64 
 7   EducationField            1470 non-null   object
 8   EmployeeCount             1470 non-null   int64 
 9   EmployeeNumber            1470 non-null   int64 
 10  EnvironmentSatisfaction   1470 non-null   int64 
 11  Gender                    1470 non-null   object
 12  HourlyRate                1470 non-null   int64 
 13  JobInvolvement            1470 non-null   int64 
 14  JobLevel                

## ----------------------------------------------------------------------------------
##### -- Employees left vs stayed

In [18]:
df['Attrition'].value_counts()

Attrition
No     1233
Yes     237
Name: count, dtype: int64

# ----------------------------------------------------------------------------------------
# 2 -- DROP UNNECESSARY COLUMNS

##### -- See all columns
##### -- Drop columns

In [22]:
df.columns

Index(['Age', 'Attrition', 'BusinessTravel', 'DailyRate', 'Department',
       'DistanceFromHome', 'Education', 'EducationField', 'EmployeeCount',
       'EmployeeNumber', 'EnvironmentSatisfaction', 'Gender', 'HourlyRate',
       'JobInvolvement', 'JobLevel', 'JobRole', 'JobSatisfaction',
       'MaritalStatus', 'MonthlyIncome', 'MonthlyRate', 'NumCompaniesWorked',
       'Over18', 'OverTime', 'PercentSalaryHike', 'PerformanceRating',
       'RelationshipSatisfaction', 'StandardHours', 'StockOptionLevel',
       'TotalWorkingYears', 'TrainingTimesLastYear', 'WorkLifeBalance',
       'YearsAtCompany', 'YearsInCurrentRole', 'YearsSinceLastPromotion',
       'YearsWithCurrManager'],
      dtype='object')

In [24]:
# Drop only useless constant columns
df = df.drop(['EmployeeCount', 'Over18', 'StandardHours'], axis=1)

#### -- Verify

In [27]:
df.shape

(1470, 32)

# -----------------------------------------------------------------------------------
# 3 -- CONVERT TARGET VARIABLE (Attrition) 
### -------------> AI models need numbers, not text 

In [30]:
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

#### -- Verify

In [33]:
df['Attrition'].value_counts()

Attrition
0    1233
1     237
Name: count, dtype: int64

# -----------------------------------------------------------------------------------
# 4 -- ENCODE CATEGORICAL VARIABLES

##### (This is mandatory before training any AI model)

In [37]:
# Right now, some columns are text-based, and ML models cannot understand text.

### 1 -- Identify Categorical Columns

In [40]:
df.select_dtypes(include='object').columns

Index(['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole',
       'MaritalStatus', 'OverTime'],
      dtype='object')

### 2 -- Encode Categorical Columns (Safe & Simple Method)
##### We will use Label Encoding (perfect for this dataset)

In [43]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in df.select_dtypes(include='object'):
    df[col] = le.fit_transform(df[col])

##### Converts text → numbers
##### Keeps data meaning intact

### 3 -- Verify Encoding Worked

In [46]:
df.head()                             # You should now see only numbers — no text

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeNumber,EnvironmentSatisfaction,...,PerformanceRating,RelationshipSatisfaction,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,1,2,1102,2,1,2,1,1,2,...,3,1,0,8,0,1,6,4,0,5
1,49,0,1,279,1,8,1,1,2,3,...,4,4,1,10,3,3,10,7,1,7
2,37,1,2,1373,1,2,2,4,4,4,...,3,2,0,7,3,3,0,0,0,0
3,33,0,1,1392,1,3,4,1,5,4,...,3,3,0,8,3,3,8,7,3,0
4,27,0,2,591,1,2,1,3,7,1,...,3,4,1,6,3,3,2,2,2,2


In [47]:
df.dtypes                           # All columns should be int or float

Age                         int64
Attrition                   int64
BusinessTravel              int32
DailyRate                   int64
Department                  int32
DistanceFromHome            int64
Education                   int64
EducationField              int32
EmployeeNumber              int64
EnvironmentSatisfaction     int64
Gender                      int32
HourlyRate                  int64
JobInvolvement              int64
JobLevel                    int64
JobRole                     int32
JobSatisfaction             int64
MaritalStatus               int32
MonthlyIncome               int64
MonthlyRate                 int64
NumCompaniesWorked          int64
OverTime                    int32
PercentSalaryHike           int64
PerformanceRating           int64
RelationshipSatisfaction    int64
StockOptionLevel            int64
TotalWorkingYears           int64
TrainingTimesLastYear       int64
WorkLifeBalance             int64
YearsAtCompany              int64
YearsInCurrent

# ---------------------------------------------------------------------------------
# 5 A -- SPLIT DATA INTO FEATURES & TARGET

In [49]:
# X → employee details
# y → attrition (0 or 1)

In [50]:
X = df.drop(['Attrition', 'EmployeeNumber'], axis=1)
y = df['Attrition']

In [51]:
# This separates input and output

# 5 B -- TRAIN–TEST SPLIT
##### The dataset was split into training and testing sets to evaluate model performance on unseen data

In [53]:
# Train model on 80% data
# Test on 20% unseen data

In [54]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 5 C -- TRAIN THE AI MODEL (Random Forest)
##### We use Random Forest because : 
#####   (i) High accuracy
#####  (ii) Handles HR data well
##### (iii) Easy to justify

In [58]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

model.fit(X_train, y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


##### Your AI model is now trained

# ----------------------------------------------------------------------
# 6 A -- MAKE PREDICTIONS ON TEST DATA

In [67]:
y_pred = model.predict(X_test)

In [70]:
# This gives Yes/No (0/1) predictions

# 6 B -- CHECK MODEL ACCURACY
##### Typical result : Around 80% – 88% (this is GOOD for HR data)
##### The model achieved good accuracy, indicating effective prediction of employee attrition

In [73]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(y_test, y_pred)
accuracy

0.8809523809523809

# 6 C -- CONFUSION MATRIX (VERY IMPORTANT)
##### This shows:
##### (i) Correct stay predictions
##### (ii) Correct attrition predictions
##### (iii) Errors (false positives / false negatives)
### Missing an employee who is about to leave is more costly than a false alarm

In [76]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
cm

array([[254,   1],
       [ 34,   5]], dtype=int64)

# 6 D -- CLASSIFICATION REPORT (OPTIONAL BUT STRONG)

In [79]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.88      1.00      0.94       255
           1       0.83      0.13      0.22        39

    accuracy                           0.88       294
   macro avg       0.86      0.56      0.58       294
weighted avg       0.88      0.88      0.84       294



In [81]:
# Focus on:
# Recall for Attrition (1)
# This matters most for HR.

# ---------- ATTRITION PROBABILITY + TOP 10 HIGH-RISK EMPLOYEES -------------

# 7 A -- PREDICT ATTRITION PROBABILITY FOR ALL EMPLOYEES
##### Instead of just 0/1, we calculate chance of leaving

In [85]:
df['Attrition_Probability'] = model.predict_proba(X)[:, 1]

In [87]:
# What this does:
# Gives probability between 0 and 1
# Higher value → higher risk of leaving

# 7 B -- VIEW ATTRITION PROBABILITY (CHECK)

In [90]:
df[['Attrition', 'Attrition_Probability']].head()

,Attrition,Attrition_Probability
0,1,0.83
1,0,0.05
2,1,0.81
3,0,0.18
4,0,0.07


# 7 C -- IDENTIFY TOP 10 EMPLOYEES LIKELY TO LEAVE

In [93]:
top_10 = df.sort_values(
    by='Attrition_Probability',
    ascending=False
).head(10)

# 7 D -- DISPLAY IMPORTANT DETAILS FOR HR

In [96]:
top_10[
    ['EmployeeNumber','Age', 'Department', 'JobRole', 'MonthlyIncome',
     'YearsAtCompany', 'JobSatisfaction',
     'WorkLifeBalance', 'Attrition_Probability']
]

,EmployeeNumber,Age,Department,JobRole,MonthlyIncome,YearsAtCompany,JobSatisfaction,WorkLifeBalance,Attrition_Probability
688,959,19,2,8,2121,1,2,4,0.95
1339,1878,22,1,6,2472,1,2,3,0.95
1332,1868,29,1,6,2439,1,4,2,0.93
1060,1494,24,1,2,3172,0,1,2,0.92
463,622,26,1,2,2340,1,4,1,0.91
127,167,19,2,8,1675,0,3,2,0.91
711,994,29,1,6,2404,0,1,3,0.90
683,952,25,2,8,2413,1,2,3,0.90
357,478,21,2,8,2174,3,2,3,0.90
457,614,18,2,8,1878,0,2,3,0.89


# 7 E -- TO GET ATTRITION PROBABILITY OF ALL EMPLOYEES IN A FILE

In [99]:
all_emp = df.sort_values(by='Attrition_Probability',ascending=False)

In [101]:
all_emp[['EmployeeNumber','Age', 'Department', 'JobRole',
         'MonthlyIncome','YearsAtCompany','JobSatisfaction','WorkLifeBalance', 
         'Attrition_Probability']]

,EmployeeNumber,Age,Department,JobRole,MonthlyIncome,YearsAtCompany,JobSatisfaction,WorkLifeBalance,Attrition_Probability
688,959,19,2,8,2121,1,2,4,0.95
1339,1878,22,1,6,2472,1,2,3,0.95
1332,1868,29,1,6,2439,1,4,2,0.93
1060,1494,24,1,2,3172,0,1,2,0.92
463,622,26,1,2,2340,1,4,1,0.91
...,...,...,...,...,...,...,...,...,...
963,1355,38,2,7,6893,7,1,3,0.00
935,1304,32,2,7,6209,10,4,4,0.00
1405,1979,31,1,5,11031,11,3,4,0.00
1404,1976,42,1,6,4332,20,3,3,0.00


# ----------------------------------- Over --------------------------------------

In [104]:
import pickle
from sklearn.preprocessing import LabelEncoder

categorical_cols = df.select_dtypes(include='object').columns

encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    df[col] = le.fit_transform(df[col])
    encoders[col] = le  # save the encoder

# Save the encoders for Streamlit app
with open("encoders.pkl", "wb") as f:
    pickle.dump(encoders, f)

# Also save your model if not done
with open("model.pkl", "wb") as f:
    pickle.dump(model, f)